# Location Mention Recognition (LMR) – Exploratory Data Analysis

## Objective
Build a system that extracts location mentions from disaster-related tweets.

---

## Dataset
- Source: Microsoft Location Mention Recognition Challenge
- Task: Extract location mentions from disaster-related tweets

## 1. Data Exploration (Structural Understanding)

In this section, we examine the raw dataset to understand its structure and content without performing any modifications.

The objective is to:
- Inspect dataset dimensions
- Review column names and data types
- Identify missing values
- Detect duplicate records
- Understand the target variable structure

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Load the training data
train_path = "../data/raw/Train_1.csv"
train = pd.read_csv(train_path)


# First few rows
train.head()

,tweet_id,text,location
0,ID_1001136212718088192,NaN,EllicottCity
1,ID_1001136696589631488,"Flash floods struck a Maryland city on Sunday,...",Maryland
2,ID_1001136950345109504,State of emergency declared for Maryland flood...,Maryland
3,ID_1001137334056833024,Other parts of Maryland also saw significant d...,Baltimore Maryland
4,ID_1001138374923579392,"Catastrophic Flooding Slams Ellicott City, Mar...",Ellicott City Maryland


In [4]:
# Dataset shape
print("Dataset Shape:", train.shape)
print("\n")

Dataset Shape: (73072, 3)




### Dataset Dimensions

The training dataset contains **73,072 rows** and **3 columns**.

Each row represents a single tweet with:
- A unique identifier (`tweet_id`)
- The tweet text (`text`)
- The labeled location mention (`location`)

In [5]:
# Column names
print("Columns:", train.columns.tolist())
print("\n")

Columns: ['tweet_id', 'text', 'location']




In [6]:
# Data types
print("Data Types:")
print(train.dtypes)
print("\n")

Data Types:
tweet_id    object
text        object
location    object
dtype: object




### 1.1 Missing Values Analysis

Before proceeding further, we examine whether there are missing values in any of the columns.  
Missing values in the `text` or `location` fields could affect modeling and may require handling during the cleaning stage.

In [7]:
train.isna().sum()

tweet_id        0
text        56624
location    29612
dtype: int64

### Missing Values Summary

- `tweet_id`: 0 missing values  
- `text`: 56,624 missing values  
- `location`: 29,612 missing values  

This indicates a substantial proportion of missing values in both the tweet text and target variable. Further investigation is required to determine whether these represent truly empty entries or formatting inconsistencies.

### 1.2 Verifying Missing Text Entries

To better understand the missing `text` values, we inspect a few rows where the text is null.

In [8]:
# Inspect rows where text is missing
train[train["text"].isna()].head(10)

,tweet_id,text,location
0,ID_1001136212718088192,NaN,EllicottCity
6,ID_1001139323075416064,NaN,Ellicott City Maryland
7,ID_1001139644023693312,NaN,NaN
8,ID_1001140017207459840,NaN,Maryland
9,ID_1001140276377935872,NaN,Maryland
10,ID_1001140804503601152,NaN,Baltimore
11,ID_1001141041926492160,NaN,Maryland
12,ID_1001141195576258560,NaN,Ellicott Maryland
13,ID_1001141289323319296,NaN,Maryland
15,ID_1001141974148239360,NaN,NaN


### Observations on Missing Text Values

Inspection of rows with missing `text` reveals:

- Several rows contain a valid `location` but no tweet text.
- Some rows contain both missing `text` and missing `location`.

From a modeling perspective, rows without tweet text cannot be used for training since there is no input data to learn from.

This issue will need to be addressed during the data cleaning stage.

### 1.3 Quantifying Usable Training Data

To understand how much data is actually usable for modeling, we calculate the number of rows where both `text` and `location` are present.

In [9]:
usable_data = train.dropna(subset=["text", "location"])

print("Total rows:", len(train))
print("Rows with both text and location:", len(usable_data))
print("Percentage usable:", round(len(usable_data) / len(train) * 100, 2), "%")

Total rows: 73072
Rows with both text and location: 11849
Percentage usable: 16.22 %


### Usable Training Data Assessment

Out of 73,072 total rows, only 11,849 rows contain both tweet text and a labeled location.

This represents approximately 16.22% of the dataset.

Implications:

- The effective supervised training dataset is significantly smaller than the raw dataset size.
- Rows missing `text` cannot be used for model training.
- Rows missing `location` cannot contribute to supervised extraction learning.
- Data cleaning will require careful filtering to retain only valid training instances.

### 1.4 Verifying Location Alignment with Text

Since this task involves extracting location mentions from tweets, it is important to verify whether the labeled `location` value appears directly within the corresponding tweet text.

If the location does not appear verbatim in the text, this may indicate normalization differences, formatting issues, or labeling inconsistencies.

In [10]:
alignment_check = usable_data.copy()
alignment_check["location_in_text"] = alignment_check.apply(
    lambda row: str(row["location"]) in str(row["text"]),
    axis=1
)

print("Percentage where location appears verbatim in text:",
      round(alignment_check["location_in_text"].mean() * 100, 2), "%")

Percentage where location appears verbatim in text: 71.3 %


#### Location–Text Alignment Analysis

Among rows containing both text and location, approximately 71.3% of the labeled locations appear verbatim within the tweet text.

Implications:

- A significant portion (28.7%) of labeled locations do not exactly match substrings in the tweet.
- This suggests potential inconsistencies such as:
  - Formatting differences (e.g., spacing or capitalization)
  - Abbreviations or normalization variations
  - Annotation discrepancies
- The extraction task may therefore require normalization handling rather than simple substring matching.

### 1.5 Duplicate Record Check

We verify whether duplicate rows exist in the dataset, as duplicates can bias model training and evaluation.

In [11]:
print("Duplicate rows:", train.duplicated().sum())

Duplicate rows: 0


#### Duplicate Record Assessment

No fully duplicated rows were detected in the dataset.

This indicates that there is no immediate risk of duplicated training instances biasing the model. However, further checks during cleaning may still examine partial duplicates (e.g., identical text with different labels).

## Data Exploration Findings Summary

Key observations from the structural exploration stage:

1. The dataset contains 73,072 rows and 3 columns.
2. Only 11,849 rows (16.22%) contain both tweet text and a labeled location.
3. A large portion of rows contain missing `text`, making them unusable for training.
4. Approximately 71.3% of labeled locations appear verbatim in the tweet text.
5. No fully duplicated rows were found.

These findings indicate that careful filtering and normalization will be required during the data cleaning stage.

## 2. Data Cleaning

Based on the structural exploration findings, the following cleaning steps will be performed:

1. Remove rows where `text` is missing.
2. Retain rows where `location` is missing, as they represent valid negative examples.
3. Apply minimal normalization to text and location fields.
4. Preserve raw data integrity by creating a cleaned copy.

All transformations are documented to ensure reproducibility.

### 2.1 Removing Rows with Missing Text

Rows without tweet text cannot be used for training since there is no input data. These rows will be removed.

In [12]:
cleaned = train.dropna(subset=["text"]).copy()

print("Original dataset size:", len(train))
print("After removing missing text:", len(cleaned))
print("Percentage retained:", round(len(cleaned)/len(train)*100, 2), "%")

Original dataset size: 73072
After removing missing text: 16448
Percentage retained: 22.51 %


#### Post-Filtering Dataset Size

After removing rows with missing tweet text, the dataset now contains 16,448 rows.

This represents 22.51% of the original dataset. The remaining rows are structurally usable for modeling, as they contain valid input text.

The next step is to examine how many of these rows contain labeled locations.

### 2.2 Target Availability After Cleaning

We now evaluate how many of the remaining rows contain a labeled location versus missing location values.

Rows with missing `location` are retained, as they represent valid negative examples.

In [13]:
print("Rows with location:", cleaned["location"].notna().sum())
print("Rows without location:", cleaned["location"].isna().sum())

print("Percentage with location:",
      round(cleaned["location"].notna().mean()*100, 2), "%")

Rows with location: 11849
Rows without location: 4599
Percentage with location: 72.04 %


#### Target Distribution After Cleaning

Among the 16,448 usable rows:

- 11,849 rows (72.04%) contain a labeled location.
- 4,599 rows (27.96%) contain no location label.

This distribution provides both positive and negative examples, which is beneficial for training a realistic extraction model capable of predicting when no location is present.

### 2.3 Basic Text Normalization

To ensure consistency without removing useful signals, minimal normalization will be applied:

- Convert text and location columns to string type.
- Strip leading and trailing whitespace.
- Normalize repeated spaces.

No aggressive transformations (e.g., lowercasing or punctuation removal) are applied at this stage.

In [14]:
# Convert to string (safe handling)
cleaned["text"] = cleaned["text"].astype(str)
cleaned["location"] = cleaned["location"].astype(str)

# Strip whitespace
cleaned["text"] = cleaned["text"].str.strip()
cleaned["location"] = cleaned["location"].str.strip()

# Normalize multiple spaces
cleaned["text"] = cleaned["text"].str.replace(r"\s+", " ", regex=True)
cleaned["location"] = cleaned["location"].str.replace(r"\s+", " ", regex=True)

cleaned.head()

,tweet_id,text,location
1,ID_1001136696589631488,"Flash floods struck a Maryland city on Sunday,...",Maryland
2,ID_1001136950345109504,State of emergency declared for Maryland flood...,Maryland
3,ID_1001137334056833024,Other parts of Maryland also saw significant d...,Baltimore Maryland
4,ID_1001138374923579392,"Catastrophic Flooding Slams Ellicott City, Mar...",Ellicott City Maryland
5,ID_1001138377717157888,WATCH: 1 missing after flash #FLOODING devasta...,Ellicott City Maryland


### 2.4 Re-evaluating Location Alignment

After applying minimal normalization, we reassess whether the labeled location appears verbatim in the corresponding tweet text.

This helps determine whether formatting inconsistencies were affecting alignment.

In [15]:
# Only check rows where location exists
alignment_check = cleaned[cleaned["location"] != "nan"].copy()

alignment_check["location_in_text"] = alignment_check.apply(
    lambda row: row["location"] in row["text"],
    axis=1
)

print("Updated percentage where location appears verbatim in text:",
      round(alignment_check["location_in_text"].mean() * 100, 2), "%")

Updated percentage where location appears verbatim in text: 71.31 %


#### Alignment After Normalization

The alignment rate remained effectively unchanged after normalization (71.31%).

This indicates that the mismatch between `location` and `text` is not caused by simple whitespace inconsistencies.

The remaining discrepancies are likely due to formatting differences, abbreviations, or annotation conventions. This will influence the modeling approach, as strict substring matching alone will not fully solve the task.

### 2.5 Finalizing Cleaned Dataset

We finalize the cleaned dataset by ensuring missing locations are properly represented as null values and saving the processed dataset for downstream modeling.

In [16]:
# Replace string "nan" back to actual NaN
cleaned["location"] = cleaned["location"].replace("nan", np.nan)

# Save cleaned dataset
cleaned.to_csv("../data/processed/train_clean.csv", index=False)

print("Cleaned dataset saved.")
print("Final dataset shape:", cleaned.shape)

Cleaned dataset saved.
Final dataset shape: (16448, 3)
